# Live slate

Set `TEAM` to an abbreviation on the latest Pinnacle board. Odds come from the newest `pinnacle_nba_*_team.json`. Starters come from the newest Rotowire file that day; a team that has already left the page keeps the last five posted earlier that day.

Layer 1 draws one game environment for that team. Layer 2 is the pregame role for each player on that team and on the opponent: his minutes, his team's form, and the opponent's form. This notebook is the live path, so those windows include every completed game before the slate. The Layer 1 draw is the script. It is not drawn again.

Layer 3 turns that role and that script into minutes. The conditional fit trains on every completed game before the slate, and each window stops before the game being scored. Simulation adds the calibration residual to the raw minute prediction, then allocates the team budget once. The live minutes are the rotation segments that realize that allocation, so the schedule does not rescale it. A segment is five players on the floor for a positive duration. Posted start, who is available, and the margin on that row choose the fives. The durations add up to the script's game minutes, so the team total is five times that length and no player plays longer than that length. A player Rotowire lists as out, or out for the season, is unavailable and gets 0 minutes. Questionable and doubtful players stay available. After the script margin is known, a posted starter is active. Everyone else is active with his close-game rate, or, when the margin is at least 20, with the chance he plays in close games or in garbage time. A script with fewer than seven active players adds the next most likely names. Inactive players get 0 minutes, and the minute budget is split across the players who are in.

Layer 4 turns those minutes and the possession counts into shots and turnovers. The model is fit on the raw logs, not on the rolling rates. The attempt pool follows the team's possessions. Both teams share one timeline, and each Layer 3 segment keeps the events with the five players on the floor. Within that five, the share follows recent attempts per minute. The live profile keeps the latest completed game, and its date has to fall after the fit. Makes and free throws then move so the players' points on that row sum to the nearest whole number of that side's scenario points.

Layer 5 credits the assists, rebounds, steals, and blocks those shots and turnovers already produced. A make can be assisted by a teammate. A miss is split between the two teams, then among the teammates who were on the floor. A steal comes from the opponent's turnovers, and a block comes from an opponent miss. The same windows decide who is credited.

Layer 6 keeps one simulation row as one box score. A prop clears when the stat is strictly above its line. The chance that two props both clear is the share of rows where both do, which is not the product of the two separate chances.

Layer 7 prices points, rebounds, assists, steals, and blocks on that box score against one book. `BOOK` starts at PrizePicks and can be set to any other saved book: underdog, novig, prophetx, pinnacle, fanduel, or draftkings. PrizePicks is quoted at -119 on both sides. For the other books the line is the two-way price whose over and under are closest. `prop_over` is the share of rows strictly above the line, and `over_ev` is the profit per dollar on that over. A push returns the stake. The five highest overs are a one-dollar-each portfolio. Its risk is the standard deviation of the combined profit on those same rows.

In [1]:
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "nba_game_simulator").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the nba_quant_analysis repo")

REPO = repo_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "nba_game_simulator" / "src"))

from game_enviroment import (
    build_features,
    build_game_table,
    load_gamelogs,
    load_latest_slate,
    load_model,
    print_sim_summary,
    save_model,
    season_of,
    simulate_game,
    train,
)
from lineup import (
    attach_lineup_fill,
    build_lineup_features,
    fill_lineup_gaps,
    merge_lineup_features,
)
from role_context import attach_script, role_for_slate
from game_boxscore import Prop, assemble_box_score, format_outcome, prop_probability
from shared_events import allocate_events, fit_events, live_events, training_events
from game_opportunities import (
    allocate_game_opportunities,
    fit_opportunities,
    live_opportunities,
    training_opportunities,
)
from game_rotations import (
    allocate_rotation,
    fit_rotation,
    invalid_segment_count,
    play_rates,
    script_exposure,
    script_rates,
    segments_from_minutes,
    training_minutes,
)
from src.live.predict_points import lineups_on
from src.pipeline.silver.rotowire import normalize_team_abbreviation
from src.scrapers.arbitrage import canonical_team

TEAM = "DEN"
MODEL_PATH = REPO / "nba_game_simulator" / "game_env_model.json"

In [2]:
log_paths = sorted((REPO / "data" / "silver" / "nba").glob("*/regular_season/player_gamelogs.parquet"))
if not log_paths:
    raise FileNotFoundError("no player gamelogs under data/silver/nba")
roster_path = log_paths[-1]
played = pd.read_parquet(
    roster_path,
    columns=["player_name", "player_id", "team_abbreviation", "game_date"],
)
played["game_date"] = pd.to_datetime(played["game_date"])
roster = (
    played.sort_values(["player_id", "game_date"])
    .groupby("player_id", as_index=False)
    .tail(1)[["player_name", "player_id", "team_abbreviation"]]
)

slate = load_latest_slate(TEAM, roster)
board = json.loads(Path(slate["pinnacle_file"]).read_text())["games"]
posted = lineups_on(slate["date"])

def abbr_for(label):
    wanted = canonical_team(label, "nba")
    for abbr in roster["team_abbreviation"].dropna().unique():
        normalized = normalize_team_abbreviation(abbr)
        if canonical_team(normalized, "nba") == wanted:
            return normalized
    return label

rows = []
for game in board:
    participants = game.get("participants") or []
    if len(participants) < 2:
        continue
    away_name, home_name = participants[0], participants[1]
    markets = game.get("team_markets") or {}
    total = None
    for block in markets.get("total") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "over" and line.get("points") is not None:
                total = line["points"]
                break
    home_spread = None
    for block in markets.get("spread") or []:
        if block.get("period") not in (0, None) or block.get("is_alternate"):
            continue
        for line in block.get("lines") or []:
            if line.get("side") == "home" and line.get("points") is not None:
                home_spread = line["points"]
                break
    home_abbr = abbr_for(home_name)
    away_abbr = abbr_for(away_name)
    rows.append(
        {
            "away": away_abbr,
            "home": home_abbr,
            "home_spread": home_spread,
            "total": total,
            "away_five": " ".join(posted.get(away_abbr) or []) or "—",
            "home_five": " ".join(posted.get(home_abbr) or []) or "—",
        }
    )

print(f"Pinnacle: {Path(slate['pinnacle_file']).name}")
print(f"Rotowire: {Path(slate['lineup_file']).name}")
print(f"roster:   {roster_path.relative_to(REPO)} through {played['game_date'].max().date()}")
display(pd.DataFrame(rows))

print(
    f"{slate['team']} vs {slate['opponent']}"
    f"  home={slate['is_home']}  spread={slate['spread']}  total={slate['total']}"
)
if slate["team_lineup"] is None or slate["opponent_lineup"] is None:
    print("A five is missing from today's Rotowire files. That side uses the training-average starters.")
else:
    print(f"starter ids  {slate['team']}: {slate['team_lineup']}")
    print(f"starter ids  {slate['opponent']}: {slate['opponent_lineup']}")
for side, missing in (
    (slate["team"], slate["team_out_absent"]),
    (slate["opponent"], slate["opponent_out_absent"]),
):
    if missing:
        print(
            f"{side} listed out with no game on this roster, so no minutes:"
            f" {', '.join(missing)}"
        )

Pinnacle: pinnacle_nba_2026-11-07_000330_team.json
Rotowire: rotowire_nba_2026-11-07_000331.json
roster:   data/silver/nba/2026-27/regular_season/player_gamelogs.parquet through 2026-11-06


,away,home,home_spread,total,away_five,home_five
0,BKN,ATL,-6.5,227.0,Nolan Traore Michael Porter Chaney Johnson Jul...,Kingston Flemings Nickeil Alexander-Walker Dys...
1,DEN,IND,7.0,229.0,Jamal Murray Christian Braun Cameron Johnson A...,Andrew Nembhard Kelly Oubre Aaron Nesmith Pasc...
2,HOU,PHX,7.0,221.5,Fred VanVleet Amen Thompson Kevin Durant Jabar...,Devin Booker Jalen Green Dillon Brooks Miles B...
3,NYK,CLE,5.0,224.0,Jalen Brunson Mikal Bridges Landry Shamet OG A...,James Harden Donovan Mitchell Peyton Watson Ev...
4,ORL,GSW,-6.5,222.5,Jase Richardson Anthony Black Desmond Bane Fra...,Stephen Curry Brandin Podziemski Yaxel Lendebo...


DEN vs IND  home=False  spread=-7.0  total=229.0
starter ids  DEN: [1627750, 1631128, 1629661, 203932, 203999]
starter ids  IND: [1629614, 1626162, 1630174, 1627783, 1627826]
DEN listed out with no game on this roster, so no minutes: Julian Strawther
IND listed out with no game on this roster, so no minutes: Johnny Furphy


In [3]:
saved = json.loads(MODEL_PATH.read_text()) if MODEL_PATH.exists() else None
if saved is not None and saved.get("version") == 3:
    model = load_model(MODEL_PATH)
    print(f"loaded {MODEL_PATH.name}  ({model.get('trained_games')} games)")
else:
    if saved is not None:
        print(
            f"{MODEL_PATH.name} is version {saved.get('version')}; "
            "retraining version 3"
        )
    raw = load_gamelogs(str(REPO / "data" / "silver" / "nba" / "*" / "regular_season" / "player_gamelogs.parquet"))
    games = build_game_table(raw)
    lineups, lineup_state = build_lineup_features(
        raw, halflife=12.0, prior_weight=4.0, season_carry=0.6, window=300,
    )
    games = merge_lineup_features(games, lineups)
    lineup_state = attach_lineup_fill(lineup_state, games)
    games = fill_lineup_gaps(games)
    games, state = build_features(
        games, halflife=12.0, prior_w=4.0, carry=0.6, window=300, min_league=30,
    )
    state["lineup"] = lineup_state
    args = argparse.Namespace(
        test_seasons=1,
        validation_season="2025-26",
        holdout_season="2026-27",
        halflife=12.0,
        prior_weight=4.0,
        season_carry=0.6,
        league_window=300,
        spread_convention="auto",
        plots=False,
    )
    model = train(games, state, args)
    save_model(model, str(MODEL_PATH))

sim = simulate_game(
    model,
    slate["team"],
    slate["opponent"],
    slate["date"],
    total=slate["total"],
    spread=slate["spread"],
    is_home=slate["is_home"],
    team_lineup=slate["team_lineup"],
    opponent_lineup=slate["opponent_lineup"],
    n=5000,
    seed=7,
)
print_sim_summary(sim, slate["total"], slate["spread"])
sim[["team_poss", "opp_poss", "pace", "team_pts", "opp_pts", "game_total", "team_margin", "n_ot"]].describe().round(2)

loaded game_env_model.json  (8367 games)

DEN at IND | total 229.0 | team spread -7.0
               mean     sd     p05     p25     p50     p75     p95
pace         100.75   3.93   94.29   98.21  100.71  103.35  107.28
team_poss    101.32   4.81   94.39   98.35  100.95  103.80  109.49
team_pts     118.20  11.45   99.40  110.45  118.06  125.84  136.79
opp_pts      111.22  11.36   92.46  103.66  111.32  119.10  129.88
game_total   229.42  18.16  199.58  217.19  229.79  241.74  258.31
team_margin    6.97  13.81  -15.58   -2.48    7.06   16.06   29.94
P(OT): 4.7%


,team_poss,opp_poss,pace,team_pts,opp_pts,game_total,team_margin,n_ot
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,101.32,101.30,100.75,118.20,111.22,229.42,6.97,0.05
std,4.81,4.82,3.93,11.45,11.36,18.16,13.81,0.26
min,85.72,85.72,85.72,79.80,66.44,162.90,-44.30,0.00
25%,98.35,98.30,98.21,110.45,103.66,217.19,-2.48,0.00
50%,100.95,100.86,100.71,118.06,111.32,229.79,7.06,0.00
75%,103.80,103.71,103.35,125.84,119.10,241.74,16.06,0.00
max,135.53,135.01,115.16,164.90,155.22,293.95,58.50,4.00


In [4]:
ROLE_COLUMNS = [
    "season_year",
    "player_id",
    "player_name",
    "team_id",
    "team_abbreviation",
    "game_id",
    "game_date",
    "matchup",
    "min",
    "pf",
    "team_pace",
    "team_off_rating",
    "team_def_rating",
    "pace",
    "off_rating",
    "def_rating",
    "start_position",
    "team_pts",
    "opp_pts",
    "player_team_spread",
    "fga",
    "fgm",
    "fg3_a",
    "fg3_m",
    "fta",
    "ftm",
    "tov",
    "team_fga",
    "team_poss",
    "team_tov",
    "ast",
    "oreb",
    "dreb",
    "stl",
    "blk",
    "team_fgm",
    "team_ast",
    "team_oreb",
    "team_stl",
    "team_blk",
    "opp_dreb",
    "opp_tov",
    "opp_fga",
]
logs = pd.concat(
    [pd.read_parquet(path, columns=ROLE_COLUMNS) for path in log_paths],
    ignore_index=True,
)
logs["game_date"] = pd.to_datetime(logs["game_date"])
slate_day = pd.Timestamp(slate["date"])
history = logs.loc[logs["game_date"] < slate_day]
if history.empty:
    raise ValueError(f"no completed games before {slate['date']}")
# Who is on the team comes from this season. The windows below still use earlier seasons.
this_season = history.loc[history["season_year"].astype(str).eq(season_of(slate_day))]
latest = (
    this_season.sort_values(["player_id", "game_date", "game_id"])
    .groupby("player_id", as_index=False)
    .tail(1)
)
latest["abbr"] = latest["team_abbreviation"].map(normalize_team_abbreviation)


def team_id_for(abbr: str) -> int:
    ids = latest.loc[latest["abbr"].eq(abbr), "team_id"].dropna()
    if ids.empty:
        raise ValueError(f"no completed games for {abbr} before {slate['date']}")
    return int(ids.mode().iloc[0])


def players_for(
    abbr, own_id, other_id, is_home, spread, starter_ids, out_ids, probable_ids
):
    """Players last seen on this team, plus anyone in the posted five.

    ``out_ids`` are Rotowire out and out-for-season players. They stay on the
    roster with ``available`` 0, so the segment schedule gives them 0 minutes.
    Probable players stay available. Posted starters are active in every script.
    """
    starters = {int(player_id) for player_id in (starter_ids or [])}
    outs = {int(player_id) for player_id in (out_ids or [])}
    probable = {int(player_id) for player_id in (probable_ids or [])}
    side = latest.loc[latest["abbr"].eq(abbr)]
    posted = latest.loc[
        latest["player_id"].astype(int).isin(starters | outs | probable)
    ]
    missing = (starters | outs | probable).difference(posted["player_id"].astype(int))
    if missing:
        print(f"{abbr} posted players missing from logs: {sorted(missing)}")
    side = pd.concat([side, posted], ignore_index=True).drop_duplicates("player_id")
    if side.empty:
        raise ValueError(f"no players for {abbr} before {slate['date']}")
    on_roster = side["player_id"].astype(int)
    return pd.DataFrame(
        {
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "team": abbr,
            "game_date": slate_day,
            "team_id": own_id,
            "opp_team_id": other_id,
            "is_home": float(bool(is_home)),
            "is_starter": on_roster.isin(starters).astype(float).to_numpy(),
            "probable": on_roster.isin(probable).astype(float).to_numpy(),
            "available": (~on_roster.isin(outs)).astype(float).to_numpy(),
            "team_spread": float(spread),
        }
    )


team_abbr = normalize_team_abbreviation(slate["team"])
opp_abbr = normalize_team_abbreviation(slate["opponent"])
team_id = team_id_for(team_abbr)
opp_id = team_id_for(opp_abbr)
players = pd.concat(
    [
        players_for(
            team_abbr,
            team_id,
            opp_id,
            slate["is_home"],
            slate["spread"],
            slate["team_lineup"],
            slate["team_out"],
            slate["team_probable"],
        ),
        players_for(
            opp_abbr,
            opp_id,
            team_id,
            not slate["is_home"],
            -float(slate["spread"]),
            slate["opponent_lineup"],
            slate["opponent_out"],
            slate["opponent_probable"],
        ),
    ],
    ignore_index=True,
)
names = players[["player_id", "player_name", "team", "available"]]
role = role_for_slate(logs, players.drop(columns=["player_name", "team"]))
role = names.merge(role, on="player_id", how="left", validate="one_to_one")
role = role.merge(
    players[["player_id", "probable"]],
    on="player_id",
    how="left",
    validate="one_to_one",
)
role["side"] = role["team"].eq(team_abbr).map({True: 0, False: 1})
role = role.sort_values(
    ["side", "is_starter", "min_mean_10"],
    ascending=[True, False, False],
).drop(columns="side")
rates = play_rates(history, role)
role["play_rate"] = rates["play_rate"].to_numpy()
recent_ids = []
for side_id in role["team_id"].drop_duplicates():
    side_games = (
        history.loc[history["team_id"].eq(side_id), ["game_id", "game_date"]]
        .drop_duplicates("game_id")
        .sort_values("game_date")
        .tail(12)["game_id"]
    )
    recent_ids.extend(side_games.astype(str))
recent_ids = set(recent_ids)
pbp = pd.read_parquet(REPO / "data" / "PlayByPlay" / "s27_pbp.parquet")
pbp = pbp.loc[pbp["gameId"].astype(str).isin(recent_ids)]
exposure_logs = history.loc[
    history["game_id"].astype(str).isin(recent_ids),
    ["game_id", "player_id", "team_id", "game_date", "min", "player_name", "start_position"],
]
script = script_rates(script_exposure(pbp, exposure_logs), role)
role["close_rate"] = script["close_rate"].to_numpy()
role["blowout_rate"] = script["blowout_rate"].to_numpy()

shown = [
    "team",
    "player_name",
    "is_starter",
    "available",
    "min_mean_10",
    "play_rate",
    "close_rate",
    "blowout_rate",
    "min_std_10",
    "start_rate_10",
    "pf_mean_10",
    "season_games_prior",
    "team_pace_10",
    "team_off_10",
    "team_def_10",
    "team_rest",
    "team_b2b",
    "opp_pace_10",
    "opp_off_10",
    "opp_def_10",
    "opp_rest",
    "opp_b2b",
    "team_spread",
]
print(
    f"live role through {history['game_date'].max().date()} for {slate_day.date()}"
    f"  ({len(role)} players, latest game included)"
)
out_now = role.loc[role["available"].eq(0), ["team", "player_name"]]
if out_now.empty:
    print("Rotowire out: none on these rosters")
else:
    listed = ", ".join(
        f"{row.team} {row.player_name}" for row in out_now.itertuples(index=False)
    )
    print(f"Rotowire out: {listed}")
display(role[shown].round(2))

team_role = role.loc[role["team"].eq(team_abbr)].reset_index(drop=True)
opp_role = role.loc[role["team"].eq(opp_abbr)].reset_index(drop=True)
opp_env = sim.copy()
opp_env["team_margin"] = -opp_env["team_margin"]
opp_env["is_home"] = 1.0 - opp_env["is_home"].astype(float)
attached = {
    team_abbr: attach_script(team_role, sim),
    opp_abbr: attach_script(opp_role, opp_env),
}
for label, frame in attached.items():
    draws = frame.drop_duplicates("sim_id")
    one = frame.loc[frame["sim_id"].eq(0), "team_final_margin"]
    if one.nunique() != 1:
        raise AssertionError(f"{label} players do not share one margin")
    print(
        f"{label}: {frame['player_id'].nunique()} players share the draw"
        f"  mean margin {draws['team_final_margin'].mean():.1f}"
        f"  P(OT) {draws['went_ot'].mean():.1%}"
        f"  P(blowout) {draws['blowout'].mean():.1%}"
    )


live role through 2026-11-06 for 2026-11-07  (30 players, latest game included)
Rotowire out: IND Tyrese Haliburton, IND Ben Sheppard


,team,player_name,is_starter,available,min_mean_10,play_rate,close_rate,blowout_rate,min_std_10,start_rate_10,...,team_off_10,team_def_10,team_rest,team_b2b,opp_pace_10,opp_off_10,opp_def_10,opp_rest,opp_b2b,team_spread
4,DEN,Jamal Murray,1.0,1.0,37.76,0.74,0.89,0.67,4.66,1.00,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
8,DEN,Christian Braun,1.0,1.0,35.15,0.74,0.89,0.67,4.75,1.00,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
2,DEN,Nikola Jokić,1.0,1.0,34.40,0.81,0.89,0.67,7.21,1.00,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
1,DEN,Aaron Gordon,1.0,1.0,30.13,0.74,0.89,0.50,5.61,1.00,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
6,DEN,Cameron Johnson,1.0,1.0,29.46,0.52,0.56,0.33,8.59,1.00,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
0,DEN,DeMar DeRozan,0.0,1.0,24.90,0.76,0.78,0.50,6.57,0.40,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
11,DEN,Trevon Brazile,0.0,1.0,22.12,0.85,0.89,0.83,5.78,0.14,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
12,DEN,Ryan Nembhard,0.0,1.0,20.01,0.48,0.11,0.50,12.78,0.60,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
13,DEN,Bryce Hopkins,0.0,1.0,18.70,0.67,0.56,0.83,11.74,0.40,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0
9,DEN,Cam Whitmore,0.0,1.0,14.81,0.43,0.25,0.50,5.12,0.00,...,119.12,114.45,3.0,0.0,102.05,109.84,113.53,3.0,0.0,-7.0


DEN: 15 players share the draw  mean margin 7.0  P(OT) 4.7%  P(blowout) 19.7%
IND: 15 players share the draw  mean margin -7.0  P(OT) 4.7%  P(blowout) 19.7%


In [5]:
trained = training_minutes(history)
rotation = fit_rotation(trained)
print(
    f"rotation fit on {len(trained):,} player-games"
    f" through {history['game_date'].max().date()}"
    "  (each window stops before that game)"
)

game_key = f"{slate_day.date()}-{team_abbr}-{opp_abbr}"
for frame in (sim, opp_env):
    frame["game_id"] = game_key
    frame["game_date"] = slate_day

rng = np.random.default_rng(7)
minutes = {
    team_abbr: allocate_rotation(
        rotation, team_role, sim, rng, method="segments"
    ),
    opp_abbr: allocate_rotation(
        rotation, opp_role, opp_env, rng, method="segments"
    ),
}
schedules = {
    team_abbr: segments_from_minutes(team_role, sim, minutes[team_abbr]),
    opp_abbr: segments_from_minutes(opp_role, opp_env, minutes[opp_abbr]),
}


def script_zero(label, side, environment, drawn):
    """One attached script, realized from that row's allocated minutes."""
    row = environment.iloc[[0]]
    game_minutes = float(row["game_minutes"].iloc[0])
    n_ot = float(row["n_ot"].iloc[0])
    margin = float(row["team_margin"].iloc[0])
    segments = schedules[label][0]
    invalid = invalid_segment_count(
        segments,
        side,
        game_minutes=game_minutes,
        n_ot=n_ot,
    )
    if invalid != 0:
        raise AssertionError(f"{label} script 0 has {invalid} invalid segments")
    total = float(drawn[0].sum())
    if abs(total - 5.0 * game_minutes) > 1e-4:
        raise AssertionError(f"{label} script 0 misses the team minute budget")
    print(
        f"{label} script 0: n_ot {int(round(n_ot))}  "
        f"team_margin {margin:+.1f}  "
        f"cap {game_minutes:.1f}  "
        f"team total {total:.1f}  "
        f"invalid segments {invalid}"
    )


script_zero(team_abbr, team_role, sim, minutes[team_abbr])
script_zero(opp_abbr, opp_role, opp_env, minutes[opp_abbr])


def minute_table(label, side, drawn):
    budget = 5.0 * sim["game_minutes"].to_numpy()
    gap = np.max(np.abs(drawn.sum(axis=1) - budget))
    if gap > 1e-4:
        raise AssertionError(f"{label} minutes miss the script budget by {gap:.4f}")
    cap = sim["game_minutes"].to_numpy()[:, None]
    if np.any(drawn > cap + 1e-6):
        raise AssertionError(f"{label} has a player above the game length")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min_mean_10": side["min_mean_10"].to_numpy(),
            "mean": drawn.mean(axis=0),
            "sd": drawn.std(axis=0),
            "p10": np.percentile(drawn, 10, axis=0),
            "p90": np.percentile(drawn, 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "mean"], ascending=[False, False])


shown_minutes = pd.concat(
    [
        minute_table(team_abbr, team_role, minutes[team_abbr]),
        minute_table(opp_abbr, opp_role, minutes[opp_abbr]),
    ],
    ignore_index=True,
)
team_means = shown_minutes.groupby("team")["mean"].sum().round(1)
print(
    f"segment minutes for {slate_day.date()}  "
    f"mean team total {team_means.to_dict()}"
)
print("sd is the spread across scripts; one script has a fixed schedule")
display(shown_minutes.round(1))

rotation fit on 179,719 player-games through 2026-11-06  (each window stops before that game)
DEN script 0: n_ot 0  team_margin +15.9  cap 48.0  team total 240.0  invalid segments 0
IND script 0: n_ot 0  team_margin -15.9  cap 48.0  team total 240.0  invalid segments 0
segment minutes for 2026-11-07  mean team total {'DEN': 241.3, 'IND': 241.3}
sd is the spread across scripts; one script has a fixed schedule


,team,player_name,is_starter,min_mean_10,mean,sd,p10,p90
0,DEN,Jamal Murray,1.0,37.8,36.8,6.2,29.2,44.7
1,DEN,Christian Braun,1.0,35.1,34.6,6.2,27.2,42.1
2,DEN,Nikola Jokić,1.0,34.4,34.0,6.1,26.5,41.6
3,DEN,Aaron Gordon,1.0,30.1,30.2,5.9,23.1,37.4
4,DEN,Cameron Johnson,1.0,29.5,29.8,6.5,21.8,37.9
5,DEN,Trevon Brazile,0.0,22.1,17.3,8.3,2.7,26.7
6,DEN,DeMar DeRozan,0.0,24.9,15.8,9.5,0.0,26.2
7,DEN,Marvin Bagley III,0.0,14.7,11.1,7.5,0.0,21.0
8,DEN,Tyus Jones,0.0,12.2,9.4,7.1,0.0,19.1
9,DEN,Bryce Hopkins,0.0,18.7,8.9,8.4,0.0,20.3


In [6]:
trained_shots = training_opportunities(history)
shot_model = fit_opportunities(history)
shot_profile = live_opportunities(
    history, players.drop(columns=["player_name", "team"])
)
shot_profile["player_id"] = shot_profile["player_id"].astype(
    players["player_id"].dtype
)
shot_profile = names.drop(columns="available").merge(
    shot_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"shot model fit on {shot_model.metadata['player_games']:,} player-games"
    f" through {shot_model.training_cutoff}"
    f"  ({len(trained_shots):,} rolling rows; each window stops before that game)"
)


def side_profile(side):
    ordered = (
        shot_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("shot profile does not match the rotation")
    return ordered


if sim["is_home"].nunique() != 1:
    raise AssertionError("the slate does not have one home flag")
slate_is_home = bool(sim["is_home"].iloc[0])
home_abbr, away_abbr = (
    (team_abbr, opp_abbr) if slate_is_home else (opp_abbr, team_abbr)
)
home_role, away_role = (
    (team_role, opp_role) if slate_is_home else (opp_role, team_role)
)
allocated = allocate_game_opportunities(
    shot_model,
    side_profile(home_role),
    side_profile(away_role),
    minutes[home_abbr],
    minutes[away_abbr],
    sim,
    rng,
    home_segments=schedules[home_abbr],
    away_segments=schedules[away_abbr],
)
shots = {home_abbr: allocated["home"], away_abbr: allocated["away"]}
for drawn in shots.values():
    if not drawn["metadata"]["segments_used"]:
        raise AssertionError("shots were not kept inside the Layer 3 segments")
for label, scenario in ((team_abbr, sim["team_pts"]), (opp_abbr, sim["opp_pts"])):
    target = np.floor(scenario.to_numpy(dtype=float) + 0.5).astype(np.int64)
    if not np.array_equal(shots[label]["PTS"].sum(axis=1), target):
        raise AssertionError(f"{label} points miss the scenario score")
for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
    unavailable = side["available"].to_numpy() == 0
    if unavailable.any() and shots[label]["FGA"][:, unavailable].sum() != 0:
        raise AssertionError(f"{label} gave a shot to an unavailable player")


def shot_table(label, side, drawn):
    if not np.array_equal(drawn["FGA"].sum(axis=1), drawn["team_fga"]):
        raise AssertionError(f"{label} attempts do not sum to the shot pool")
    if not np.array_equal(drawn["TOV"].sum(axis=1), drawn["team_tov"]):
        raise AssertionError(f"{label} turnovers do not sum to the turnover pool")
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "fga_per_min": side_profile(side)["fga_per_min"].to_numpy(),
            "fga": drawn["FGA"].mean(axis=0),
            "pts": drawn["PTS"].mean(axis=0),
            "tov": drawn["TOV"].mean(axis=0),
            "pts_p10": np.percentile(drawn["PTS"], 10, axis=0),
            "pts_p90": np.percentile(drawn["PTS"], 90, axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_shots = pd.concat(
    [
        shot_table(team_abbr, team_role, shots[team_abbr]),
        shot_table(opp_abbr, opp_role, shots[opp_abbr]),
    ],
    ignore_index=True,
)
attempt_means = {
    team_abbr: round(float(shots[team_abbr]["team_fga"].mean()), 1),
    opp_abbr: round(float(shots[opp_abbr]["team_fga"].mean()), 1),
}
print(f"live shots for {slate_day.date()}  mean team attempts {attempt_means}")
display(shown_shots.round(2))


/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:38: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:82: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  frame = _prepare_opportunity_logs(logs)
/Users/alexgonzalez/Documents/nba_quant_analysis/nba_game_simulator/src/game_opportunities/train.py:180: UserWarning: zero-minute rows have positive statistics; per-minute denominators exclude them
  prepared = _prepare_opportunity_logs(logs)


shot model fit on 179,719 player-games through 2026-11-06  (179,719 rolling rows; each window stops before that game)
live shots for 2026-11-07  mean team attempts {'DEN': 87.3, 'IND': 87.3}


,team,player_name,is_starter,fga_per_min,fga,pts,tov,pts_p10,pts_p90
0,DEN,Nikola Jokić,1.0,0.48,16.28,21.82,3.17,13.0,31.0
1,DEN,Jamal Murray,1.0,0.41,15.13,21.48,2.68,12.0,31.0
2,DEN,Christian Braun,1.0,0.31,10.77,14.44,1.55,7.0,22.0
3,DEN,Aaron Gordon,1.0,0.33,9.73,13.31,1.47,6.0,21.0
4,DEN,Cameron Johnson,1.0,0.33,9.70,12.72,1.53,5.0,21.0
5,DEN,DeMar DeRozan,0.0,0.40,6.33,8.34,1.00,0.0,17.0
6,DEN,Trevon Brazile,0.0,0.30,5.18,7.05,0.90,0.0,14.0
7,DEN,Marvin Bagley III,0.0,0.35,3.90,5.60,0.64,0.0,13.0
8,DEN,Bryce Hopkins,0.0,0.28,2.50,3.65,0.55,0.0,10.0
9,DEN,Tyus Jones,0.0,0.30,2.79,3.39,0.56,0.0,9.0


In [7]:
trained_events = training_events(history)
event_model = fit_events(history)
event_profile = live_events(
    history, players.drop(columns=["player_name", "team"])
)
event_profile["player_id"] = event_profile["player_id"].astype(
    players["player_id"].dtype
)
event_profile = names.drop(columns="available").merge(
    event_profile, on="player_id", how="left", validate="one_to_one"
)
print(
    f"event model fit through {event_model.cutoff}"
    f"  ({len(trained_events):,} rolling rows; each window stops before that game)"
)


def event_side(side):
    ordered = (
        event_profile.set_index("player_id").loc[side["player_id"]].reset_index()
    )
    if len(ordered) != len(side) or not ordered["player_id"].equals(side["player_id"]):
        raise AssertionError("event profile does not match the rotation")
    return ordered


availability = {
    team_abbr: team_role["available"].to_numpy() == 1,
    opp_abbr: opp_role["available"].to_numpy() == 1,
}
events, event_details = allocate_events(
    event_model,
    {team_abbr: event_side(team_role), opp_abbr: event_side(opp_role)},
    shots,
    minutes,
    rng,
    segments=schedules,
    availability=availability,
    return_details=True,
)
if not event_details["segments_used"]:
    raise AssertionError("events were not kept inside the Layer 3 segments")
for label, side in ((team_abbr, team_role), (opp_abbr, opp_role)):
    if event_details["sources"][label]["AST"] != "layer4":
        raise AssertionError(f"{label} assists were drawn again")
    if event_details["sources"][label]["OREB"] != "layer4":
        raise AssertionError(f"{label} offensive rebounds were drawn again")
    if not np.array_equal(events[label]["AST"], shots[label]["AST"]):
        raise AssertionError(f"{label} assists are not the Layer 4 assists")
    if not np.array_equal(events[label]["OREB"], shots[label]["OREB"]):
        raise AssertionError(f"{label} offensive rebounds are not the Layer 4 rebounds")
    if not np.array_equal(events[label]["REB"], events[label]["OREB"] + events[label]["DREB"]):
        raise AssertionError(f"{label} rebounds are not offensive plus defensive")
    unavailable = side["available"].to_numpy() == 0
    if unavailable.any():
        for stat in ("AST", "OREB", "DREB", "STL", "BLK"):
            if events[label][stat][:, unavailable].sum() != 0:
                raise AssertionError(f"{label} gave {stat} to an unavailable player")
for label, other in ((team_abbr, opp_abbr), (opp_abbr, team_abbr)):
    misses = shots[label]["FGA"].sum(axis=1) - shots[label]["FGM"].sum(axis=1)
    rebounds = events[label]["OREB"].sum(axis=1) + events[other]["DREB"].sum(axis=1)
    if np.any(rebounds > misses):
        raise AssertionError(f"rebounds on {label} misses exceed those misses")
    if np.any(events[other]["STL"].sum(axis=1) > shots[label]["TOV"].sum(axis=1)):
        raise AssertionError(f"steals exceed {label} turnovers")
    if np.any(events[other]["BLK"].sum(axis=1) > misses):
        raise AssertionError(f"blocks on {label} shots exceed its misses")


def event_table(label, side, drawn):
    summary = pd.DataFrame(
        {
            "team": label,
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "ast_per_min": event_side(side)["ast_per_min"].to_numpy(),
            "ast": drawn["AST"].mean(axis=0),
            "reb": drawn["REB"].mean(axis=0),
            "stl": drawn["STL"].mean(axis=0),
            "blk": drawn["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "ast"], ascending=[False, False])


shown_events = pd.concat(
    [
        event_table(team_abbr, team_role, events[team_abbr]),
        event_table(opp_abbr, opp_role, events[opp_abbr]),
    ],
    ignore_index=True,
)
assist_means = {
    team_abbr: round(float(events[team_abbr]["AST"].sum(axis=1).mean()), 1),
    opp_abbr: round(float(events[opp_abbr]["AST"].sum(axis=1).mean()), 1),
}
print(f"live events for {slate_day.date()}  mean team assists {assist_means}")
display(shown_events.round(2))


event model fit through 2026-11-06  (179,719 rolling rows; each window stops before that game)
live events for 2026-11-07  mean team assists {'DEN': 26.3, 'IND': 25.0}


,team,player_name,is_starter,ast_per_min,ast,reb,stl,blk
0,DEN,Nikola Jokić,1.0,0.24,5.87,9.52,2.09,0.87
1,DEN,Jamal Murray,1.0,0.16,5.16,5.42,1.02,0.65
2,DEN,Aaron Gordon,1.0,0.10,3.12,5.58,1.03,0.41
3,DEN,Cameron Johnson,1.0,0.09,2.79,5.06,1.19,0.33
4,DEN,Christian Braun,1.0,0.07,2.54,4.77,1.89,0.26
5,DEN,DeMar DeRozan,0.0,0.11,1.68,1.77,0.56,0.15
6,DEN,Tyus Jones,0.0,0.14,1.15,1.21,0.38,0.11
7,DEN,Trevon Brazile,0.0,0.04,0.98,4.23,0.72,0.36
8,DEN,Marvin Bagley III,0.0,0.07,0.90,2.64,0.30,0.30
9,DEN,Bryce Hopkins,0.0,0.06,0.67,1.44,0.26,0.07


In [12]:
box = assemble_box_score(
    sim,
    {"home": home_role["player_id"], "away": away_role["player_id"]},
    {"home": minutes[home_abbr], "away": minutes[away_abbr]},
    {"home": shots[home_abbr], "away": shots[away_abbr]},
    {"home": events[home_abbr], "away": events[away_abbr]},
    # Layer 4 already locked points with half-up rounding. Ties round differently here.
    check_score=False,
)
if box.n_sims != len(sim):
    raise AssertionError("the box score does not have one row per script")
for label, side_name in ((home_abbr, "home"), (away_abbr, "away")):
    stats = box.teams[side_name].stats
    if not np.array_equal(stats["PTS"], shots[label]["PTS"]):
        raise AssertionError(f"{label} box points are not the Layer 4 points")
    if not np.allclose(stats["MIN"], minutes[label]):
        raise AssertionError(f"{label} box minutes are not the rotation minutes")
    if not np.array_equal(stats["AST"], events[label]["AST"]):
        raise AssertionError(f"{label} box assists are not the Layer 5 assists")
    if not np.array_equal(stats["REB"], events[label]["REB"]):
        raise AssertionError(f"{label} box rebounds are not the Layer 5 rebounds")


def box_table(label, side, side_name):
    stats = box.teams[side_name].stats
    summary = pd.DataFrame(
        {
            "team": label,
            "player_id": side["player_id"].to_numpy(),
            "player_name": side["player_name"].to_numpy(),
            "is_starter": side["is_starter"].to_numpy(),
            "min": stats["MIN"].mean(axis=0),
            "pts": stats["PTS"].mean(axis=0),
            "fgm": stats["FGM"].mean(axis=0),
            "fga": stats["FGA"].mean(axis=0),
            "fg3m": stats["3PM"].mean(axis=0),
            "fg3a": stats["3PA"].mean(axis=0),
            "ftm": stats["FTM"].mean(axis=0),
            "fta": stats["FTA"].mean(axis=0),
            "reb": stats["REB"].mean(axis=0),
            "ast": stats["AST"].mean(axis=0),
            "stl": stats["STL"].mean(axis=0),
            "blk": stats["BLK"].mean(axis=0),
        }
    )
    return summary.sort_values(["is_starter", "pts"], ascending=[False, False])


shown_box = pd.concat(
    [
        box_table(home_abbr, home_role, "home"),
        box_table(away_abbr, away_role, "away"),
    ],
    ignore_index=True,
)
print(f"box score for {slate_day.date()}  {box.n_sims:,} rows, one game each")
display(shown_box.drop(columns="player_id").round(2))

played = pd.DataFrame(
    format_outcome(
        box,
        0,
        stats=(
            "MIN",
            "PTS",
            "REB",
            "AST",
            "STL",
            "BLK",
            "FGM",
            "FGA",
            "3PM",
            "3PA",
            "FTM",
            "FTA",
        ),
    )
)
played = played.rename(columns={"3PM": "FG3M", "3PA": "FG3A"})
lookup = pd.concat(
    [
        home_role.assign(side="home"),
        away_role.assign(side="away"),
    ],
    ignore_index=True,
)
lookup["player"] = lookup["player_id"].map(lambda value: str(int(value)))
lookup["team"] = lookup["side"]
lookup["team_name"] = lookup["side"].map({"home": home_abbr, "away": away_abbr})
played = played.merge(
    lookup[["team", "player", "player_name", "team_name"]],
    on=["team", "player"],
    how="left",
    validate="one_to_one",
)
played["team"] = played["team_name"]
played = played.loc[
    played["MIN"] > 0,
    [
        "team",
        "player_name",
        "MIN",
        "PTS",
        "REB",
        "AST",
        "STL",
        "BLK",
        "FGM",
        "FGA",
        "FG3M",
        "FG3A",
        "FTM",
        "FTA",
    ],
]
print(f"simulation 0 for {slate_day.date()}")
display(played.reset_index(drop=True))

leaders = (
    shown_box.loc[shown_box["team"].eq(team_abbr)]
    .sort_values(["is_starter", "pts"], ascending=[False, False])
    .head(2)
)
slate_side = "home" if team_abbr == home_abbr else "away"
props = []
for row in leaders.itertuples(index=False):
    index = list(box.teams[slate_side].player_ids).index(str(int(row.player_id)))
    column = box.teams[slate_side].stats["PTS"][:, index]
    props.append(Prop(slate_side, row.player_id, "PTS", float(np.floor(np.median(column)) + 0.5)))
priced = prop_probability(box, props)
print(
    f"P({leaders.iloc[0].player_name} over {props[0].line} and "
    f"{leaders.iloc[1].player_name} over {props[1].line})"
    f" = {priced.joint:.3f}; product of the separate chances = {priced.independent_product:.3f}"
)
print(f"separate chances = {tuple(round(value, 3) for value in priced.marginals)}")


box score for 2026-11-07  5,000 rows, one game each


,team,player_name,is_starter,min,pts,fgm,fga,fg3m,fg3a,ftm,fta,reb,ast,stl,blk
0,IND,Pascal Siakam,1.0,30.57,19.31,7.48,14.79,1.32,4.02,3.03,4.18,6.58,3.19,1.09,0.14
1,IND,Andrew Nembhard,1.0,32.46,15.72,5.27,11.91,2.00,5.55,3.19,3.76,2.83,6.03,0.75,0.21
2,IND,Aaron Nesmith,1.0,27.95,13.71,4.76,10.59,1.94,5.38,2.25,2.87,3.40,2.16,1.04,0.28
3,IND,Kelly Oubre Jr.,1.0,26.59,11.55,4.34,9.48,1.20,3.94,1.67,2.19,4.16,1.90,1.28,0.52
4,IND,Ivica Zubac,1.0,23.79,9.69,3.87,8.08,0.38,1.07,1.58,2.14,6.54,1.99,0.36,0.52
5,IND,Quenton Jackson,0.0,17.61,8.29,2.87,6.33,0.84,2.53,1.71,2.03,2.75,2.13,1.09,0.21
6,IND,Obi Toppin,0.0,12.47,7.44,2.68,5.46,0.92,2.42,1.15,1.45,2.79,1.01,0.27,0.13
7,IND,Jarace Walker,0.0,18.64,6.96,2.50,6.08,0.99,2.83,0.97,1.38,4.38,1.69,0.56,0.30
8,IND,Jay Huff,0.0,13.69,5.67,2.06,4.45,0.78,2.37,0.77,0.96,2.85,1.18,0.20,0.50
9,IND,Jalen Slawson,0.0,12.58,3.93,1.24,2.90,0.38,1.25,1.07,1.44,2.48,1.23,0.75,0.36


simulation 0 for 2026-11-07


,team,player_name,MIN,PTS,REB,AST,STL,BLK,FGM,FGA,FG3M,FG3A,FTM,FTA
0,IND,Andrew Nembhard,36.624999,18,4,4,3,2,7,18,2,7,2,3
1,IND,Pascal Siakam,35.838539,15,12,4,0,0,6,15,1,3,2,4
2,IND,Aaron Nesmith,27.560175,14,7,3,1,0,5,10,3,4,1,3
3,IND,Kelly Oubre Jr.,37.865642,13,10,4,3,1,5,27,0,14,3,6
4,IND,Ivica Zubac,26.900120,10,11,2,1,0,4,8,0,2,2,3
5,IND,Quenton Jackson,24.484945,11,8,2,0,1,4,9,3,6,0,2
6,IND,Jay Huff,13.845213,1,2,0,0,2,0,3,0,3,1,1
7,IND,Kobe Brown,21.157966,12,3,2,0,1,4,8,3,4,1,1
8,IND,Larry Nance Jr.,15.722401,2,6,4,0,0,1,4,0,2,0,1
9,DEN,Jamal Murray,48.000000,19,12,7,1,0,7,22,2,9,3,3


P(Nikola Jokić over 21.5 and Jamal Murray over 21.5) = 0.232; product of the separate chances = 0.237
separate chances = (0.497, 0.477)


In [15]:
BOOK = "fanduel"

from game_boxscore import portfolio_risk, price_over
from src.live.prop_markets import attach_roster, book_names, latest_prop_markets

markets, market_audit = latest_prop_markets(REPO / "data" / "props", BOOK)
others = ", ".join(name for name in book_names() if name != market_audit["book"])
print(
    f"Layer 7 {market_audit['book']} from {market_audit['source_file']}. "
    f"Change BOOK to {others}."
)
if market_audit["book"] == "prizepicks":
    print("PrizePicks has no two-way price. Both sides are -119.")

roster = pd.concat(
    [
        home_role.assign(team=home_abbr, side="home"),
        away_role.assign(team=away_abbr, side="away"),
    ],
    ignore_index=True,
)
offered = attach_roster(
    markets,
    roster.loc[:, ["team", "side", "player_id", "player_name"]],
)
slate_dates = sorted(set(pd.to_datetime(offered["slate_date"]).dt.date))
if offered.empty:
    print(f"no {market_audit['book']} lines matched this game")
elif slate_dates != [slate_day.date()]:
    print(f"market dates {slate_dates}; the slate date is {slate_day.date()}")


def stat_board(stat, label):
    rows = []
    ranked = []
    for offer in offered.loc[offered["stat"].eq(stat)].itertuples(index=False):
        index = list(box.teams[offer.side].player_ids).index(offer.player_id)
        minutes = box.teams[offer.side].stats["MIN"][:, index]
        column = box.teams[offer.side].stats[stat][:, index]
        priced = price_over(
            box,
            Prop(offer.side, offer.player_id, stat, float(offer.line)),
            float(offer.over_american),
        )
        rows.append(
            {
                "team": offer.team,
                "player_name": offer.player_name,
                "min": float(minutes.mean()),
                label: float(column.mean()),
                f"{label}_line": float(offer.line),
                "odds": int(offer.over_american),
                "prop_over": priced.p_over,
                "over_ev": priced.ev,
            }
        )
        ranked.append(
            {
                "team": offer.team,
                "player_name": offer.player_name,
                "stat": label,
                "min": float(minutes.mean()),
                "line": float(offer.line),
                "odds": int(offer.over_american),
                "prop_over": priced.p_over,
                "over_ev": priced.ev,
                "side": offer.side,
                "player_id": offer.player_id,
                "box_stat": stat,
            }
        )
    board = pd.DataFrame(rows)
    if not board.empty:
        board = board.sort_values("over_ev", ascending=False)
    return board, pd.DataFrame(ranked)


ranked_rows = []
for stat, label in (("PTS", "pts"), ("REB", "reb"), ("AST", "ast"), ("STL", "stl"), ("BLK", "blk")):
    board, ranked = stat_board(stat, label)
    if not ranked.empty:
        ranked_rows.append(ranked)
    print(f"{label} overs for {slate_day.date()}")
    if board.empty:
        print(f"no {label} line for this game")
        continue
    shown = board.copy()
    shown["min"] = shown["min"].round(2)
    shown[label] = shown[label].round(2)
    shown["prop_over"] = shown["prop_over"].round(3)
    shown["over_ev"] = shown["over_ev"].round(3)
    display(shown.reset_index(drop=True))

if not ranked_rows:
    print("no priced props for this game")
else:
    ranked = pd.concat(ranked_rows, ignore_index=True)
    top = ranked.sort_values("over_ev", ascending=False).head(5).reset_index(drop=True)
    print(f"top {len(top)} overs by expected value")
    shown_top = top.drop(columns=["side", "player_id", "box_stat"]).copy()
    shown_top["min"] = shown_top["min"].round(2)
    shown_top["prop_over"] = shown_top["prop_over"].round(3)
    shown_top["over_ev"] = shown_top["over_ev"].round(3)
    display(shown_top)
    legs = [
        Prop(row.side, row.player_id, row.box_stat, float(row.line))
        for row in top.itertuples(index=False)
    ]
    held = portfolio_risk(box, legs, top["odds"].tolist())
    print(
        f"portfolio of these {held.n}, $1 on each over. "
        f"expected profit = {held.expected_profit:.3f}; "
        f"risk = {held.risk:.3f}; "
        f"P(profit < 0) = {held.p_loss:.3f}; "
        f"P(all clear) = {held.joint:.3f}; "
        f"product of the separate chances = {held.independent_product:.3f}"
    )


Layer 7 fanduel from fanduel_nba_2026-11-07_000332.json. Change BOOK to prizepicks, underdog, novig, prophetx, pinnacle, draftkings.
pts overs for 2026-11-07


,team,player_name,min,pts,pts_line,odds,prop_over,over_ev
0,IND,Andrew Nembhard,32.46,15.72,3.5,-112,0.985,0.865
1,DEN,Aaron Gordon,30.21,13.31,2.5,-112,0.984,0.863
2,IND,Kelly Oubre Jr.,26.59,11.55,9.5,-112,0.612,0.158
3,DEN,Christian Braun,34.61,14.44,12.5,-112,0.604,0.144
4,IND,Aaron Nesmith,27.95,13.71,12.5,-104,0.543,0.064
5,DEN,Jamal Murray,36.81,21.48,22.5,-110,0.426,-0.187
6,IND,Ivica Zubac,23.79,9.69,10.5,-102,0.399,-0.209
7,IND,Pascal Siakam,30.57,19.31,20.5,-120,0.416,-0.238
8,IND,Obi Toppin,12.47,7.44,11.5,-112,0.292,-0.447
9,DEN,DeMar DeRozan,15.84,8.34,12.5,102,0.265,-0.464


reb overs for 2026-11-07


,team,player_name,min,reb,reb_line,odds,prop_over,over_ev
0,IND,Kelly Oubre Jr.,26.59,4.16,3.5,102,0.543,0.098
1,DEN,Jamal Murray,36.81,5.42,4.5,-112,0.565,0.070
2,DEN,Christian Braun,34.61,4.77,4.5,-118,0.483,-0.108
3,DEN,Aaron Gordon,30.21,5.58,5.5,-112,0.460,-0.129
4,IND,Aaron Nesmith,27.95,3.40,3.5,102,0.423,-0.145
5,IND,Pascal Siakam,30.57,6.58,6.5,-130,0.460,-0.186
6,IND,Andrew Nembhard,32.46,2.83,2.5,-172,0.506,-0.199
7,DEN,DeMar DeRozan,15.84,1.77,2.5,120,0.294,-0.354
8,IND,Ivica Zubac,23.79,6.54,8.5,118,0.262,-0.429
9,DEN,Nikola Jokić,33.96,9.52,11.5,-138,0.292,-0.496


ast overs for 2026-11-07


,team,player_name,min,ast,ast_line,odds,prop_over,over_ev
0,DEN,Aaron Gordon,30.21,3.12,2.5,-112,0.594,0.124
1,DEN,Christian Braun,34.61,2.54,2.5,136,0.467,0.103
2,IND,Pascal Siakam,30.57,3.19,3.5,-144,0.390,-0.340
3,DEN,Jamal Murray,36.81,5.16,6.5,104,0.270,-0.450
4,IND,Andrew Nembhard,32.46,6.03,7.5,102,0.269,-0.456
5,DEN,DeMar DeRozan,15.84,1.68,2.5,-158,0.287,-0.532
6,DEN,Nikola Jokić,33.96,5.87,9.5,-112,0.081,-0.847


stl overs for 2026-11-07
no stl line for this game
blk overs for 2026-11-07
no blk line for this game
top 5 overs by expected value


,team,player_name,stat,min,line,odds,prop_over,over_ev
0,IND,Andrew Nembhard,pts,32.46,3.5,-112,0.985,0.865
1,DEN,Aaron Gordon,pts,30.21,2.5,-112,0.984,0.863
2,IND,Kelly Oubre Jr.,pts,26.59,9.5,-112,0.612,0.158
3,DEN,Christian Braun,pts,34.61,12.5,-112,0.604,0.144
4,DEN,Aaron Gordon,ast,30.21,2.5,-112,0.594,0.124


portfolio of these 5, $1 on each over. expected profit = 2.153; risk = 1.683; P(profit < 0) = 0.075; P(all clear) = 0.226; product of the separate chances = 0.213
